# Lesson 3 — The Ghost Fluid Method in 1-D

Prereq: Lesson 2 (jump conditions). Reference: Paper 1 §3.

Goal: see why a naive Poisson solver **smears** a jump across the flame front, and how GFM gives a **sharp** solution using the same matrix structure. We'll build both in 1-D with `ρ = 1` for clarity, jumping `[p]` at a front at position `x = x_f`.

Model problem:

$$-p''(x) = f(x), \quad x \in [0,1], \quad p(0)=p(1)=0$$

with the requirement that `p` has a prescribed jump `[p] = p⁺ − p⁻` at `x = x_f`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve

N = 80
h = 1.0 / N
x = (np.arange(N) + 0.5) * h        # cell centers
x_f = 0.5 + 0.37 * h                 # front location, deliberately off a cell face
jump_p = 1.0                         # [p] = p(x_f+) - p(x_f-)
f = np.zeros(N)                      # no body source

# Exact analytic solution:  two linear pieces, zero at both ends, jump at x_f.
def exact(x):
    left  = (-jump_p * x_f) * x / x_f                # = -jump_p * x on x < x_f? no, pick constants
    # Solve -p'' = 0 on each side with p(0)=0, p(1)=0, and p(x_f+) - p(x_f-) = jump_p,
    # with continuous p' (no body force jump).
    # => p = A*x on left, p = B*(x-1) on right.
    # Continuity of p': A = B.  Jump: B*(x_f-1) - A*x_f = jump_p.
    # With A = B: A*(x_f - 1 - x_f) = jump_p  =>  A = -jump_p.
    A = -jump_p
    return np.where(x < x_f, A*x, A*(x - 1))

p_exact = exact(x)

## 1. The naive Poisson solve

Standard 3-point stencil `-p''_i ≈ (-p_{i-1} + 2 p_i - p_{i+1}) / h² = f_i`, ignoring the jump entirely.

In [ ]:
# Tridiagonal Laplacian with Dirichlet BC (p = 0 at ghost cells just outside the domain)
def build_laplacian(N, h):
    main = 2.0 * np.ones(N) / h**2
    off  = -1.0 * np.ones(N-1) / h**2
    A = diags([off, main, off], [-1, 0, 1], format='lil')
    return A

A_naive = build_laplacian(N, h)
rhs_naive = f.copy()
# Dirichlet: p = 0 outside domain, absorbed into the -1/h² ghost contribution = 0.
p_naive = spsolve(A_naive.tocsr(), rhs_naive)

## 2. GFM: put the jump into the RHS

Say the front sits between cell `i = i_f` (left of front, fuel) and cell `i_f + 1` (right of front, hot). Let `θ` be the fraction of the edge `[i_f, i_f+1]` on the fuel side.

The stencil that needs fixing is the two stencils that cross the front:

- Row `i_f` uses `p_{i_f+1}` as if it were continuous with `p_{i_f}`. Subtract the jump: `p_{i_f+1}^{used} = p_{i_f+1}^{true} - [p]`.
- Row `i_f + 1` uses `p_{i_f}` the same way, but now the jump has opposite sign: `p_{i_f}^{used} = p_{i_f}^{true} + [p]`.

Both corrections move to the RHS. Matrix is unchanged.

In [ ]:
# Locate front: largest i with cell-center x[i] < x_f
i_f = np.searchsorted(x, x_f) - 1
print(f'Front between cells {i_f} (x={x[i_f]:.4f}) and {i_f+1} (x={x[i_f+1]:.4f}); x_f = {x_f:.4f}')

A_gfm = build_laplacian(N, h)       # same matrix!
rhs_gfm = f.copy()
# Row i_f:  Laplacian expects (−p_{i_f+1} + 2 p_{i_f} − p_{i_f-1})/h² = f_{i_f}
# We pretend p_{i_f+1} lives on fuel side by subtracting [p]. That moves +[p]/h² to RHS.
rhs_gfm[i_f]    += -jump_p / h**2
# Row i_f+1: stencil uses p_{i_f}; pretend it lives on hot side by adding [p]. Moves −[p]/h² to RHS.
rhs_gfm[i_f+1]  +=  jump_p / h**2

p_gfm = spsolve(A_gfm.tocsr(), rhs_gfm)

## 3. Compare

In [ ]:
plt.figure(figsize=(9, 5))
xx = np.linspace(0, 1, 400)
plt.plot(xx, exact(xx), 'k-', lw=2, label='exact (with jump)')
plt.plot(x, p_naive, 'o--', color='tab:red', label='naive Poisson (smears)')
plt.plot(x, p_gfm,   's:',  color='tab:blue', label='GFM (sharp)')
plt.axvline(x_f, color='gray', lw=0.7, ls='--', label='front $x_f$')
plt.legend()
plt.xlabel('x'); plt.ylabel('p')
plt.title('Naive vs GFM pressure across a jump')
plt.grid(alpha=0.3)
plt.show()

err_naive = np.abs(p_naive - p_exact).max()
err_gfm   = np.abs(p_gfm   - p_exact).max()
print(f'L∞ error, naive: {err_naive:.4f}')
print(f'L∞ error, GFM:   {err_gfm:.4f}')

Expect the naive solve to average the jump away — zero everywhere because the RHS is zero and no boundary forces a step. The GFM solve matches the exact two-piece linear solution up to first-order error at the front.

**Takeaway:**
- **Matrix unchanged** — same tridiagonal Laplacian, same CG solver would work.
- **RHS gets `±[p]/h²` on two cells** straddling the front.
- In 3-D and with position-dependent density, you also need `1/ρ` coefficients and a `θ`-weighted harmonic average at front-crossing faces, but the structure is the same.

That's the whole ghost-fluid idea for pressure. The velocity-jump `[V·n]` enters the RHS the same way, through the divergence term.

## 4. Refinement study (optional)

Convergence rate of GFM should be `O(h)` at the jump and `O(h²)` away from it. Run at multiple N and plot error vs h.

In [ ]:
Ns = [40, 80, 160, 320, 640]
errs = []
for N in Ns:
    h = 1.0 / N
    x = (np.arange(N) + 0.5) * h
    x_f_local = 0.5 + 0.37 * h
    i_f = np.searchsorted(x, x_f_local) - 1
    A = build_laplacian(N, h)
    rhs = np.zeros(N)
    rhs[i_f]   += -jump_p / h**2
    rhs[i_f+1] +=  jump_p / h**2
    p = spsolve(A.tocsr(), rhs)
    p_ex = np.where(x < x_f_local, -jump_p*x, -jump_p*(x-1))
    errs.append(np.abs(p - p_ex).max())

plt.figure(figsize=(6, 4))
plt.loglog(Ns, errs, 'o-', label='GFM max error')
plt.loglog(Ns, [errs[0]*Ns[0]/N for N in Ns], 'k--', label=r'$O(h)$ reference')
plt.xlabel('N'); plt.ylabel('L∞ error'); plt.legend(); plt.grid(which='both', alpha=0.3)
plt.title('GFM convergence (first order at the jump)')
plt.show()

## Exercises

1. Change `jump_p` to a negative number and re-solve. The pressure step flips sign as expected.
2. Add a nonzero body source `f = sin(πx)` and confirm GFM still gets it right.
3. (Harder) Make `ρ` jump too: `ρ = 1` on the left, `ρ = 0.125` on the right. The LHS matrix now has `1/ρ` coefficients that differ per face. At the front-crossing face use the `θ`-weighted harmonic average `ρ̂ = ρ_f ρ_h / (θ ρ_h + (1−θ) ρ_f)`. This is the 2-D/3-D setup, stripped to 1-D.